# Covariance-model comparisons


Computational scripts save assignment `-1` first, then random assignments `0–99`.
`PAIRING_VIEW` changes only the loaded/displayed results. Full results exclude `-1`
and give each random assignment equal weight after averaging its internal folds.
Mean ± SD are descriptive, not confidence intervals. Missing assignments are an
error; assignment zero and incomplete batches are never substituted for the full result.
Raw signal preparation runs once in a terminal and includes every eligible MEG subject.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
PAIRING_VIEW = 'matched'  # 'matched': -1; 'full': mean ± SD of IDs 0–99.
N_PAIRINGS = 100
PAIRING_SEED = 2026
from vs0_run_pairings import activate, format_command, pairing_context
activate(PAIRING_VIEW, N_PAIRINGS, PAIRING_SEED, display_only=True)
import vs0_load_and_config as shared_data
from vs0_pairing_results import config_view, safe_load
ANALYSIS_OUTPUT_ROOT = shared_data.analysis_output_root(ROOT)
DISPLAY_COMPONENTS = 5
SEED = 2026  # Trial/source-control seed, independent of PAIRING_SEED.
BLOCK_SCALING = True
N_COMPONENTS = 10
CONDITION_MODE = 'average'
MAX_GRAM_GIB = 2.
from vs1_coverage_figures import MODALITY_COLORS, SETUP_COLORS
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .25,
                     'axes.spines.top': False, 'axes.spines.right': False})

MEG_KIND='paired_coverage'
OUTPUT_DIR=ANALYSIS_OUTPUT_ROOT/'covariance'
MODEL_LINESTYLES = {'separate_pca': '-', 'joint_pca': '--', 'plssvd': ':'}


## Configuration and terminal preparation

In [2]:
MEG_DIR = shared_data.MEG_DATA_DIR
IEEG_DIR = ROOT/'ieeg_shortWOBS_fs250'
METADATA_CSV = None
PROJECT_PATH = None
SFREQ = 250.
MEG_TMIN = None
MEG_TIMES_FILE = None
CORRECT_MEG_ZSCORE = True
CORRECT_IEEG_X1000 = True
CONDITIONS = [1,2]
CONFIG = dict(data=dict(meg_dir=str(MEG_DIR), ieeg_dir=str(IEEG_DIR),
    metadata_csv=str(METADATA_CSV) if METADATA_CSV else None,
    project_path=str(PROJECT_PATH) if PROJECT_PATH else None,
    meg_coordinate_unit='m', ieeg_coordinate_unit='mm', meg_tmin=MEG_TMIN,
    meg_scaling='channel_zscore' if CORRECT_MEG_ZSCORE else 'none',
    ieeg_scaling='none', ieeg_multiplier=1000. if CORRECT_IEEG_X1000 else 1.,
    meg_times_file=str(MEG_TIMES_FILE) if MEG_TIMES_FILE else None,
    include_all_meg=True, sfreq=SFREQ, conditions=CONDITIONS,
    prepared_cache_dir=str(ROOT/'out'/'prepared_data')),
    output_dir=str(OUTPUT_DIR), block_scaling=BLOCK_SCALING, condition_mode=CONDITION_MODE,
    n_components=N_COMPONENTS, seed=SEED, pairing_seed=PAIRING_SEED,
    n_pairings=N_PAIRINGS, max_gram_gib=MAX_GRAM_GIB)

CONFIG['meg_kind']=MEG_KIND

In [3]:
# This cell writes configuration and prints a terminal command; no raw signals are prepared here.
CONFIG_FILE = OUTPUT_DIR/'config.json'
CONFIG_FILE.parent.mkdir(parents=True, exist_ok=True)
CONFIG_FILE.write_text(json.dumps(CONFIG, indent=2))
import shlex
print(shlex.join(['python','-u',str(ROOT/'vs0_prepare_data.py'),'--config',str(CONFIG_FILE)]))

python -u /users/barbara/Desktop/MIB_iEEG/iEEGvsMEG/vs0_prepare_data.py --config /users/barbara/Desktop/MIB_iEEG/iEEGvsMEG/out/subject_pairings/9b08db76e28b/iteration_-1/covariance/config.json


## Fit and load all saved results

In [4]:
RECOMPUTE=False
command=['python','-u',str(ROOT/'vs2_run_cov_models.py'),'--config',str(CONFIG_FILE)]
if RECOMPUTE: command.append('--recompute')
print(format_command(command))
models_view=safe_load(config_view, CONFIG, 'models', PAIRING_VIEW, N_PAIRINGS)

python -u /users/barbara/Desktop/MIB_iEEG/iEEGvsMEG/vs0_run_pairings.py --view matched --scope all --n-pairings 100 --seed 2026 -- python -u /users/barbara/Desktop/MIB_iEEG/iEEGvsMEG/vs2_run_cov_models.py --config /users/barbara/Desktop/MIB_iEEG/iEEGvsMEG/out/subject_pairings/9b08db76e28b/iteration_-1/covariance/config.json
Analysis is not complete: /users/barbara/Desktop/MIB_iEEG/iEEGvsMEG/out/subject_pairings/9b08db76e28b/iteration_-1/covariance/models/6df949e2427aab71
Run the printed terminal command, then reload this cell.


## Variance explained, shared covariance, native matching and fixed-PCA-reference metrics

Each assignment is fitted before aggregation. Reconstruction variance and cross-covariance energy have different denominators and must not be added. Component pair identities remain labels, not averaged indices.

In [5]:
if models_view is not None:
    unit = ('saved match (-1)' if models_view.view == 'matched' else
            f'{len(models_view.assignment_ids)} random assignments; mean ± sample SD')
    print(unit)
    for name, summary in models_view.tables.items():
        if not summary['metrics']:
            continue
        print(name)
        report = summary['mean'].copy()
        if models_view.view == 'full':
            for metric in summary['metrics']:
                report[metric + '_std'] = summary['std'][metric]
                report[metric + '_n'] = summary['count'][metric]
        display(report)

        x = next((key for key in summary['keys'] if key in
                  ('k', 'component', 'n_sources', 'n_channels', 'source_count',
                   'n_selected', 'complexity')), None)
        if x is None:
            continue
        groups = [key for key in summary['keys'] if key != x]
        mean, sd = summary['mean'], summary['std']
        indices = (mean.groupby(groups, dropna=False).indices if groups else
                   {'all': np.arange(len(mean))})
        for metric in summary['metrics']:
            if len(indices) > 24 or metric.startswith('angles_deg_'):
                continue  # These metrics remain available in the complete table.
            fig, ax = plt.subplots(figsize=(10, 3.5), layout='constrained')
            for label, rows in indices.items():
                rows = np.asarray(rows)[np.argsort(mean.iloc[rows][x].to_numpy())]
                frame = mean.iloc[rows]
                labels = label if isinstance(label, tuple) else (label,)
                setup = next((str(value) for value in labels if str(value) in SETUP_COLORS), MEG_KIND)
                model = next((str(value) for value in labels if str(value) in MODEL_LINESTYLES), None)
                ax.errorbar(frame[x], frame[metric],
                            yerr=sd.iloc[rows][metric] if models_view.view == 'full' else None,
                            color=SETUP_COLORS[setup], linestyle=MODEL_LINESTYLES.get(model, '-'),
                            marker='o', markersize=3, capsize=2, label=str(label))
            ax.set(xlabel=x, ylabel=metric, title=f'{name}: {unit}')
            ax.legend(fontsize=6)
            plt.show()


## Time courses

Full scores are aligned to the matched fit from fitted temporal scores. Separate PCA is aligned independently by modality; joint PCA and PLSSVD retain coupled signs and permutations.

In [6]:
if models_view is not None:
    unit = ('saved match (-1)' if models_view.view == 'matched' else
            f'{len(models_view.assignment_ids)} random assignments; mean ± sample SD')
    for name, summary in models_view.arrays.items():
        if 'scores' not in name:
            continue
        mean, sd = summary['mean'], summary['std']
        times = (models_view.times if models_view.times is not None and
                 len(models_view.times) == len(mean) else np.arange(len(mean)))
        model = name.split('/')[0]
        color = MODALITY_COLORS['iEEG'] if 'ieeg' in name else SETUP_COLORS[MEG_KIND]
        for component in range(min(DISPLAY_COMPONENTS, mean.shape[1])):
            fig, ax = plt.subplots(figsize=(10, 3), layout='constrained')
            ax.plot(times, mean[:, component], color=color,
                    linestyle=MODEL_LINESTYLES.get(model, '-'))
            if models_view.view == 'full':
                ax.fill_between(times, mean[:, component] - sd[:, component],
                                mean[:, component] + sd[:, component], color=color, alpha=.2)
            ax.set(title=f'{name}, aligned component {component + 1}: {unit}',
                   xlabel='Time (s)' if times is models_view.times else 'Observation', ylabel='Score')
            plt.show()


## Covariance structure, component variances and signed correlations

Every matrix shows its mean and a separate SD panel in full view. SD is across assignment-level matrices. All matrices below use matched-reference component correspondence.

In [7]:
if models_view is not None:
    for name, summary in models_view.arrays.items():
        if 'scores' in name:
            continue
        mean, sd = summary['mean'], summary['std']
        if mean.ndim >= 2:
            for index in np.ndindex(mean.shape[:-2]):
                matrix, matrix_sd = mean[index], sd[index]
                fig, axes = plt.subplots(1, 2 if models_view.view == 'full' else 1,
                                         figsize=(11, 4), squeeze=False, layout='constrained')
                # Signed means use a symmetric scale; SD is nonnegative.
                limit = np.nanmax(np.abs(matrix)) if np.isfinite(matrix).any() else 1.
                limit = limit or 1.
                is_correlation = 'correlation' in name
                image = axes[0, 0].imshow(matrix, aspect='auto', cmap='RdBu_r',
                                          vmin=-1 if is_correlation else -limit,
                                          vmax=1 if is_correlation else limit)
                axes[0, 0].set(title=f"{name} {index}: {'Mean' if models_view.view == 'full' else 'Matched'}",
                               xlabel='Component', ylabel='Component')
                fig.colorbar(image, ax=axes[0, 0])
                if models_view.view == 'full':
                    image_sd = axes[0, 1].imshow(matrix_sd, aspect='auto', cmap='viridis', vmin=0)
                    axes[0, 1].set(title=f'{name} {index}: SD', xlabel='Component', ylabel='Component')
                    fig.colorbar(image_sd, ax=axes[0, 1])
                plt.show()
        elif mean.ndim == 1:
            color = MODALITY_COLORS['iEEG'] if 'ieeg' in name else SETUP_COLORS[MEG_KIND]
            model = name.split('/')[0]
            fig, ax = plt.subplots(figsize=(8, 4), layout='constrained')
            ax.errorbar(np.arange(1, len(mean) + 1), mean,
                        yerr=sd if models_view.view == 'full' else None, color=color,
                        linestyle=MODEL_LINESTYLES.get(model, '-'), marker='o', capsize=2)
            ax.set(title=f'{name}: {models_view.view}', xlabel='Component', ylabel='Value')
            plt.show()
        elif mean.ndim == 0:
            print(name)
            display(pd.DataFrame(dict(mean=[float(mean)], std=[float(sd)], count=[int(summary['count'])])))


## Within-modality model comparisons

The tables above include all subspace, individually matched component, and full component-correlation metrics, with assignment SD and valid counts. Native scalar spectra/rank metrics retain native order; aligned score/matrix displays use the matched reference.